# Ukrainian primary unlearning: prepare, run and inspect

This portable notebook uses an **external frozen release**. Set `FAIRFORGET_RELEASE`
to its absolute directory. Preparation is the default mode; the training cell
runs only when `RUN_SELECTED_TRAINING` is explicitly enabled in an allocated
JUPITER GPU session with the qualified training runtime.

The current integration profile is Qwen3-8B, Ukrainian, seed 42, four target
training families and two development families, with AI review and pending
human validation. The budget is at most 10 epochs and 128 optimizer steps.
The configured 0.05-nat task-vector development KL gate is an engineering
check. These inputs do not support confirmatory or broad utility claims.
Actual training and evaluation evidence comes from saved execution receipts.

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

release_value = os.environ.get("FAIRFORGET_RELEASE")
RELEASE = Path(release_value).expanduser().resolve() if release_value else None
RUN_SELECTED_TRAINING = False
SELECTED_METHOD = "wga_kl"
TRAINING_PYTHON = os.environ.get("FAIRFORGET_TRAINING_PYTHON", sys.executable)
OUTPUT = os.environ.get("FAIRFORGET_OUTPUT")

if RELEASE is None:
    print("Preparation: set FAIRFORGET_RELEASE to an existing frozen release to validate recipes.")
elif not RELEASE.is_dir():
    print(f"Preparation: release directory is unavailable: {RELEASE}")
else:
    print("External release:", RELEASE)
print("Selected method:", SELECTED_METHOD)
print("Run GPU training:", RUN_SELECTED_TRAINING)

## Five distinct contracts

All gradient streams are reviewed Ukrainian inputs. Correction, preservation,
development and locked evaluation keep separate roles. Parameter types, masks,
objective reductions and precision policies are recorded in the recipes.

In [ ]:
METHODS = {
    "wga_kl": (
        "Detached token-confidence weighted ascent",
        "Frozen-base token KL; no corrective CE",
    ),
    "npo_kl": (
        "Token-mean completion NPO likelihood ratio",
        "Frozen-base token KL; no corrective CE",
    ),
    "rmu": (
        "One fixed seeded representation target",
        "Frozen-base activation MSE on genuine retain text",
    ),
    "biasunlearn": (
        "NPO + anti-stereotype CE + adversarial subset + dev pool controller",
        "Separate reference KL preservation",
    ),
    "task_vector": (
        "Positive completion CE, then negative scaling of acquired LoRA B",
        "Development scale selection under the declared retain gate",
    ),
}
display(
    Markdown(
        "| Method | Intervention | Preservation |\n| --- | --- | --- |\n"
        + "\n".join(
            f"| `{method}` | {parts[0]} | {parts[1]} |" for method, parts in METHODS.items()
        )
    )
)

## Validate the frozen recipes

The maintained validator checks source hashes, authorized gradient IDs,
Ukrainian stream roles, family separation, required objective settings and local
checkpoint identity. The release must contain `src/`, `recipes/`, reviewed
`data/`, the evaluator and separate evaluation/qualification manifests.
Run this cell in the qualified training environment when the release is
available. It reads local files and performs no model inference or downloads.

In [ ]:
validation = {}
validated_configs = {}
if RELEASE is not None and RELEASE.is_dir():
    sys.path.insert(0, str(RELEASE / "src"))
    try:
        from fairforget.unlearning import trainer as primary_trainer

        if not Path(primary_trainer.__file__).resolve().is_relative_to((RELEASE / "src").resolve()):
            raise RuntimeError("Restart the kernel: another release source is already imported.")
        validate_config = primary_trainer.validate_config

        for method in METHODS:
            try:
                config, streams, provenance = validate_config(
                    RELEASE / "recipes" / f"{method}.json", methods={method}
                )
                validated_configs[method] = config
                validation[method] = {
                    "status": "validated",
                    "stream_rows": {role: len(rows) for role, rows in streams.items()},
                    "max_steps": config["training"]["max_steps"],
                    "model_revision": config["model_revision"],
                }
            except (ValueError, FileNotFoundError, KeyError) as error:
                validation[method] = {"status": "failed", "reason": str(error)}
    except (ImportError, RuntimeError) as error:
        print("Recipe validation pending:", error)
else:
    print("Recipe validation pending: external release is not available.")
print(json.dumps(validation, ensure_ascii=False, indent=2))

## Run one selected method

Set `FAIRFORGET_OUTPUT` to a new result directory outside the frozen release.
Use the qualified JUPITER training kernel, or point `FAIRFORGET_TRAINING_PYTHON`
to its Python executable. The command below uses the exact validated recipe;
`RUN_SELECTED_TRAINING=False` keeps preparation mode. The trainer itself
requires CUDA and refuses existing nonempty outputs.

For the full four-GPU queue and runtime-proof enforcement, use the scheduler
launchers in [the deployment guide](../../deploy/unlearning_primary/README.md).
Raw datasets and model weights stay in the external release/model store.

In [ ]:
training_command = None
if SELECTED_METHOD not in METHODS:
    raise ValueError("Select one of the five declared methods.")
if RELEASE is not None and OUTPUT:
    selected_output = Path(OUTPUT).expanduser().resolve()
    if selected_output.is_relative_to(RELEASE):
        raise ValueError("Training output must be outside the frozen release.")
    module = (
        "fairforget.unlearning.rmu" if SELECTED_METHOD == "rmu" else "fairforget.unlearning.trainer"
    )
    output_flag = "--output-dir" if SELECTED_METHOD == "rmu" else "--output"
    training_command = [
        TRAINING_PYTHON,
        "-m",
        module,
        "--config",
        str(RELEASE / "recipes" / f"{SELECTED_METHOD}.json"),
        output_flag,
        str(selected_output),
    ]
    print("PYTHONPATH=" + shlex.quote(str(RELEASE / "src")), shlex.join(training_command))
else:
    print("Command pending: set FAIRFORGET_RELEASE and FAIRFORGET_OUTPUT.")

if RUN_SELECTED_TRAINING:
    if training_command is None or validation.get(SELECTED_METHOD, {}).get("status") != "validated":
        raise RuntimeError("A validated recipe and a new external output path are required.")
    environment = {
        **os.environ,
        "PYTHONPATH": str(RELEASE / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""),
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "PYTHONDONTWRITEBYTECODE": "1",
    }
    subprocess.run(training_command, env=environment, check=True)
else:
    print("Preparation mode: no GPU training started.")

## Inspect actual execution receipts

For an orchestrated run, optionally set `FAIRFORGET_TRAINING` to the directory
containing `execution.json` and `FAIRFORGET_EVALUATION` to the directory
containing `evaluation_execution.json`. Training completion, native collection,
paired analysis and successful mitigation are separate states. Available-arm
collection after a partial failure does not establish all-five-method completion.

In [ ]:
def read_receipt(environment_key, filename):
    value = os.environ.get(environment_key)
    if not value:
        print(f"Receipt pending: set {environment_key}.")
        return None
    path = Path(value).expanduser() / filename
    if not path.is_file():
        print("Receipt unavailable:", path)
        return None
    return json.loads(path.read_text())


training_receipt = read_receipt("FAIRFORGET_TRAINING", "execution.json")
if training_receipt is not None:
    print("Training stage:", training_receipt.get("stage"))
    display(
        Markdown(
            "| Method | Recorded status | Actual steps | Checkpoint |\n| --- | --- | ---: | --- |\n"
            + "\n".join(
                f"| `{method}` | {record.get('status')} | {record.get('steps', '')} | {record.get('checkpoint', '')} |"
                for method, record in training_receipt.get("methods", {}).items()
            )
        )
    )

evaluation_receipt = read_receipt("FAIRFORGET_EVALUATION", "evaluation_execution.json")
if evaluation_receipt is not None:
    print("Native collection stage:", evaluation_receipt.get("stage"))
    print("Recorded checkpoints:", list(evaluation_receipt.get("checkpoints", {})))
    print("No-update qualification:", evaluation_receipt.get("no_update"))
    print(json.dumps(evaluation_receipt.get("tasks", {}), indent=2))

## Native evaluation and paired analysis

The native evaluator runs separately from training and uses its own qualified
runtime. Use `deploy/unlearning_primary/evaluate.sbatch` for the full queue, or
`evaluate_checkpoint.sh RELEASE CHECKPOINT NEW_OUTPUT [TASK]` inside a GPU
allocation for one checkpoint. The external full panel contains StereoSet-UK,
BBQ-UK, both WinoBias variants, WinoPron-UK and WarBias triplets.

The repeated no-update qualification uses the separate development-only
manifest. Its rows do not count as locked-panel completion. Final scientific
effects must use the saved native outputs, matched checkpoint/input identities
and the maintained Fair-UK paired analysis. Inspect native reports and exposure
records before interpreting any optimizer-loss change as mitigation.